# run_dbt — Install dbt and execute `dbt build` against a target

This notebook is designed to be triggered by a Databricks Job (see Part 18).
The `target` parameter controls which `profiles.yml` output is used:
`dev`, `ci`, or `prod`.

**Prerequisites:**
- Databricks Secrets scope `aw` populated with `host`, `http_path`, `dbt_token`.
- The repo cloned as a Databricks Git folder at the path used in Cell 6.

In [ ]:
# ── Cell 2: Parameters ────────────────────────────────────────────────────────
# When triggered by a Job, the Job passes `target` as a widget parameter.
# When run interactively, the default "prod" is used unless overridden.
dbutils.widgets.text("target", "prod")
target = dbutils.widgets.get("target") or "prod"
print(f"dbt target: {target}")

In [ ]:
# ── Cell 3: Install dbt ───────────────────────────────────────────────────────
# Pin to the same minor version used locally (see Part 3).
%pip install dbt-core==1.8.* dbt-databricks==1.8.*

In [ ]:
# ── Cell 4: Restart Python after %pip install ─────────────────────────────────
# Required so the newly installed packages are importable in subsequent cells.
dbutils.library.restartPython()

In [ ]:
# ── Cell 5: Re-read the target widget after restart ───────────────────────────
# restartPython() clears all in-memory state, so we re-read the widget.
target = dbutils.widgets.get("target") or "prod"
print(f"dbt target (post-restart): {target}")

In [ ]:
# ── Cell 6: Inject secrets as environment variables ───────────────────────────
# Secrets are read from the 'aw' Databricks Secret scope.
# Values are NEVER printed — Databricks redacts them automatically.
# To create the scope, run from your local terminal (see Part 18 §3):
#   databricks secrets create-scope aw
#   databricks secrets put-secret aw host        --string-value "adb-xxxx..."
#   databricks secrets put-secret aw http_path   --string-value "/sql/1.0/..."
#   databricks secrets put-secret aw dbt_token   --string-value "dapi..."
import os
os.environ["DBT_DBX_HOST"]           = dbutils.secrets.get(scope="aw", key="host")
os.environ["DBT_DBX_HTTP_PATH"]      = dbutils.secrets.get(scope="aw", key="http_path")
os.environ["DBT_DBX_HTTP_PATH_PROD"] = dbutils.secrets.get(scope="aw", key="http_path")
os.environ["DBT_DBX_TOKEN"]          = dbutils.secrets.get(scope="aw", key="dbt_token")
os.environ["DBT_TARGET"]             = target
print("Secrets loaded. Running against target:", target)

In [ ]:
# ── Cell 7: Run dbt ───────────────────────────────────────────────────────────
# %sh inherits the environment variables set in the previous Python cell.
# Adjust the REPO path to match your Git folder location in the workspace.
%sh
set -e   # exit immediately on any error — causes the Job task to fail cleanly

REPO="/Workspace/Users/<you>@example.com/adventureworks-databricks-medallion-dbt"
cd "$REPO"

echo "==> dbt deps"
dbt deps

echo "==> dbt source freshness (target: $DBT_TARGET)"
dbt source freshness --target "$DBT_TARGET"

echo "==> dbt build (target: $DBT_TARGET)"
dbt build --target "$DBT_TARGET"

echo "==> Done."